In [2]:
import pandas as pd

df = pd.read_csv(
    '../data/raw/online_retail_II.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
print(df.shape)

(1067371, 8)


2.isnull() / isna() — har cell True/False batata hai (missing hai ya nahi):

In [3]:
print(df.isnull())          # poori DataFrame True/False mein (bahut bada output, sirf samajhne ke liye)
print(df.isnull().sum())    # har column ka total missing count — yeh zyada useful hai

         Invoice  StockCode  Description  Quantity  InvoiceDate  Price  \
0          False      False        False     False        False  False   
1          False      False        False     False        False  False   
2          False      False        False     False        False  False   
3          False      False        False     False        False  False   
4          False      False        False     False        False  False   
...          ...        ...          ...       ...          ...    ...   
1067366    False      False        False     False        False  False   
1067367    False      False        False     False        False  False   
1067368    False      False        False     False        False  False   
1067369    False      False        False     False        False  False   
1067370    False      False        False     False        False  False   

         Customer ID  Country  
0              False    False  
1              False    False  
2              

3.Missing values ka percentage nikalna (zyada meaningful hota hai raw count se):

In [4]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct.round(2))

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64


4.notnull() — opposite check (kaunsi values present hain):

In [5]:
print(df.notnull().sum())   # har column mein kitni non-missing values hain

Invoice        1067371
StockCode      1067371
Description    1062989
Quantity       1067371
InvoiceDate    1067371
Price          1067371
Customer ID     824364
Country        1067371
dtype: int64


5.Sirf missing wale columns dikhana (filter karke, saaf output ke liye):

In [6]:
missing_summary = df.isnull().sum()
print(missing_summary[missing_summary > 0])

Description      4382
Customer ID    243007
dtype: int64


6.Row-level missing check — kaunsi rows mein kitne columns missing hain:

In [7]:
df['missing_count_per_row'] = df.isnull().sum(axis=1)
print(df['missing_count_per_row'].value_counts())

# Woh rows dikhao jinme 1 se zyada column missing hai
multiple_missing = df[df['missing_count_per_row'] > 1]
print(multiple_missing.shape)

missing_count_per_row
0    824364
1    238625
2      4382
Name: count, dtype: int64
(4382, 9)


7."Gande" data ki pehchaan — sirf NULL nahi, balki suspicious values bhi dhoondo:

In [8]:
# Negative Quantity (returns/cancellations ka signal)
negative_qty = df[df['Quantity'] < 0]
print(f"Negative quantity rows: {negative_qty.shape[0]}")

# Zero ya negative Price (galat data ho sakta hai)
zero_negative_price = df[df['Price'] <= 0]
print(f"Zero/negative price rows: {zero_negative_price.shape[0]}")
print(zero_negative_price[['StockCode', 'Description', 'Price']].head(10))

# Empty string description (NULL nahi, lekin blank)
empty_desc = df[df['Description'].str.strip() == '']
print(f"Empty string descriptions: {empty_desc.shape[0] if not empty_desc.empty else 0}")

Negative quantity rows: 22950
Zero/negative price rows: 6207
     StockCode   Description  Price
263      21733  85123a mixed    0.0
283      71477         short    0.0
284     85123A   21733 mixed    0.0
470      21646           NaN    0.0
3114     20683           NaN    0.0
3161     21350           NaN    0.0
3162     35956          lost    0.0
3168    35605A       damages    0.0
3731     84292           NaN    0.0
4296     18010           NaN    0.0
Empty string descriptions: 0


8.Duplicate rows check karna (Day 34 mein remove karenge, aaj sirf identify):

In [9]:
duplicate_count = df.duplicated().sum()
print(f"Fully duplicate rows: {duplicate_count}")
print(df[df.duplicated(keep=False)].sort_values('Invoice').head(10))

Fully duplicate rows: 34335
    Invoice StockCode                        Description  Quantity  \
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
394  489517     21912           VINTAGE SNAKES & LADDERS         1   
391  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
390  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
388  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
386  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   

            InvoiceDate  Price  Customer ID         Country  \
362 2009-12-01 11:34:00   3.75      16329.0  United Kingdom   
394 2009-12-01 11:34:00   3.75      16329.0  United Kingdom   
391 2009-12-

9.Summary report banana (README ke liye bhi useful):

In [10]:
print("=== Data Quality Summary ===")
print(f"Total rows: {len(df)}")
print(f"Missing Customer ID: {df['Customer ID'].isnull().sum()} ({df['Customer ID'].isnull().mean()*100:.2f}%)")
print(f"Missing Description: {df['Description'].isnull().sum()} ({df['Description'].isnull().mean()*100:.2f}%)")
print(f"Negative Quantity rows: {(df['Quantity'] < 0).sum()}")
print(f"Zero/negative Price rows: {(df['Price'] <= 0).sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Cancellation invoices (start with C): {df['Invoice'].str.startswith('C', na=False).sum()}")

=== Data Quality Summary ===
Total rows: 1067371
Missing Customer ID: 243007 (22.77%)
Missing Description: 4382 (0.41%)
Negative Quantity rows: 22950
Zero/negative Price rows: 6207
Duplicate rows: 34335
Cancellation invoices (start with C): 19494


Practice questions:

1.Pata karo kya missing Customer ID wale rows mein Country ka koi specific pattern hai (jaise sirf ek country zyada aati hai).

In [13]:
# Missing Customer ID wale rows ko filter karo
missing_cust = df[df['Customer ID'].isnull()]

# Country ke hisaab se count aur percentage nikalo
country_pattern = missing_cust['Country'].value_counts()
country_pattern_pct = missing_cust['Country'].value_counts(normalize=True) * 100

# Combined summary DataFrame bana kar print karo
summary_df = pd.DataFrame({
    'Missing_Count': country_pattern,
    'Percentage (%)': country_pattern_pct.round(2)
})
print(summary_df.head(10))

                      Missing_Count  Percentage (%)
Country                                            
United Kingdom               240029           98.77
EIRE                           1671            0.69
Hong Kong                       364            0.15
Unspecified                     232            0.10
France                          128            0.05
Switzerland                     125            0.05
Portugal                        116            0.05
United Arab Emirates            114            0.05
Bahrain                          67            0.03
Israel                           47            0.02


2.Price <= 0 wale rows ka StockCode dekho — kya yeh woh hi special codes hain (POST, DOT, ADJUST, M) jo Day 32 mein discuss honge?

In [14]:
# Price <= 0 ya zero/negative price wale rows
zero_negative_price = df[df['Price'] <= 0]

# Unke StockCode ka frequency distribution dekho
print(zero_negative_price['StockCode'].value_counts().head(20))

StockCode
46000M    18
22501     18
79321     17
21116     16
22423     16
23084     16
46000S    15
22734     15
22139     14
35965     14
84990     13
22502     13
20713     13
22469     12
84016     11
22627     11
71477     10
22470     10
22625     10
22624     10
Name: count, dtype: int64
